# Spark Session

In [1]:
import os, pickle, glob
from pyspark.sql import SparkSession

In [2]:
# spark session builder
spark = SparkSession.builder \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "10g") \
    .config('spark.executor.instances', 15) \
    .getOrCreate()

In [3]:
spark

In [4]:
import requests
import pandas as pd

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
spark_ui = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
spark_ui['maxMemory_GB'] = (spark_ui['maxMemory'] / (1024**3)).round(2)
spark_ui

,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,15,1099746508,0,True,1.02


# download data using Kaggle API

In [4]:
### ONLY RUN ONCE TO DOWNLOAD DATASET ###
### commented out to prevent accidental re-download

# os.environ['KAGGLE_USERNAME'] = 'gkao'
# os.environ['KAGGLE_KEY'] = 'KGAT_bc5d318dc0aa98be982de71ce37e678c'

# from kaggle import api # import the already authenticated API client

# # 4 csv files, 27.91GB -> downloaded
# api.dataset_download_files('noahpersaud/reddit-submissions-dec-2022-to-feb-2023', path='shared', unzip=True)

# # 15 csv files, 105.67GB -> downloaded
# api.dataset_download_files('noahpersaud/reddit-submissions-july-2021-to-oct-2022', path='shared', unzip=True)

In [5]:
df = spark.read.csv('shared/', header=True)
df.describe()

DataFrame[summary: string, title: string, post_id: string, over_18: string, subreddit: string, link_flair_text: string, self_text: string]

In [4]:
df.show(5)

+--------------------+-------+-------+--------------------+---------------+--------------------+
|               title|post_id|over_18|           subreddit|link_flair_text|           self_text|
+--------------------+-------+-------+--------------------+---------------+--------------------+
|Pokémon is SO 199...| tt4ppj|  false|   TheFriendlyHermit|           NULL|           [removed]|
|Looking for peopl...| ttdavv|  false|  MakeNewFriendsHere|           NULL|           [deleted]|
|VIDEO Zelensky sp...| ttdavw|  false|             ukraine|            WAR|                NULL|
|first knife in ma...| ttdavy|  false|              knives|           NULL|"Amazon will deli...|
|come on, what do ...| ttdavz|  false|IThinkYouShouldLeave|           NULL|                NULL|
+--------------------+-------+-------+--------------------+---------------+--------------------+
only showing top 5 rows

